# 主動型投資- 資料爬蟲
**以「週」為單位做台股的買賣，賺價差**  
數據(yahoo finance上個股的資訊)：
*  日期(每周最後交易日)
*  股價(每週最後交易日當天收盤價)
*  大戶(三大法人)買賣超張數
*  大戶籌碼比例
*  借券賣出餘額樟張數
*  融資餘額張數
*  券資比
*  融資維持率

投資操作：
* 買_時機點：大戶籌碼比例增加 & 大戶借券量下降 & 融資(韭菜)餘額持平/下降，買進價格<=大戶平均成本
* 賣_時機點：大戶籌碼比例下降 & 大戶借券量增加 & 融資(韭菜)餘額持平/上升 & 融資維持率持平/下降，賣出價格>=韭菜平均成本



## 取得資料
數據來源url (ABBR為台股代碼；若為上櫃/興櫃股票，url中的TW要改為TWO)：
* 法人買賣超：https://tw.finance.yahoo.com/quote/ABBR.TW/institutional-trading (要選則「週」)
* 資券變化：https://tw.finance.yahoo.com/quote/ABBR.TW/margin (要選擇「資券餘額/借券賣出餘額」&「週」)
* 大戶籌碼：https://tw.finance.yahoo.com/quote/ABBR.TW/major-holders

### 匯入Package

In [1]:
import asyncio
import re
import sys
import threading
import time
from datetime import date, datetime
from pathlib import Path
from statistics import median

import pandas as pd
from playwright.async_api import async_playwright

### 爬蟲函式定義(共用、法人買賣超、資券餘額/借券賣出餘額、大戶籌碼)

In [7]:
# ===== 共用：變數與函式 =====
# (輸入代碼、開網頁、點按鈕、確認週資料、讀表格、跑 Playwright、存檔；下方各資料的爬蟲都會用到)

URL_TEMPLATE = "https://tw.finance.yahoo.com/quote/{symbol}/{page}"
SUFFIXES = (".TW", ".TWO")  # 上市用 .TW；上櫃、興櫃用 .TWO
SELECTED_BG = "rgb(218, 237, 255)"  # 按鈕被選取時的淡藍底色 (#daedff)
PERIOD_BUTTONS = ("日", "週", "月", "季")


def ask_stock_code():
    """請使用者輸入 4 位數字的台股代碼"""
    while True:
        abbr = input("請輸入台股代碼(4位數字)：").strip()
        if re.fullmatch(r"\d{4}", abbr):
            return abbr
        print(f"「{abbr}」不是 4 位數字，請重新輸入。")


def looks_weekly(dates):
    """用表格日期判斷是否為「週」資料：每列落在不同週，且相鄰兩列間隔的中位數約 7 天"""
    if len(dates) < 3:
        return False
    gaps = [(a - b).days for a, b in zip(dates, dates[1:])]
    iso_weeks = [d.isocalendar()[:2] for d in dates]
    return all(g > 0 for g in gaps) and len(set(iso_weeks)) == len(iso_weeks) and 5 <= median(gaps) <= 9


async def _launch_browser(p, headless):
    # 優先使用 Windows 內建的 Edge，失敗再用 Playwright 自帶的 Chromium (需先執行 playwright install chromium)
    try:
        return await p.chromium.launch(channel="msedge", headless=headless)
    except Exception:
        return await p.chromium.launch(headless=headless)


async def _open_section(page, abbr, page_name, heading, timeout_s):
    """依序嘗試 .TW、.TWO 網址，回傳 (實際代號, 標題含 heading 且有表格的區塊)
    page_name 是網址最後一段，例如 institutional-trading、margin、major-holders"""
    tried = []
    for suffix in SUFFIXES:
        symbol = abbr + suffix
        url = URL_TEMPLATE.format(symbol=symbol, page=page_name)
        tried.append(url)
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout_s * 1000)

        # Yahoo 有時會自動導正代號，以頁面標題上的代號為準，例如「環球晶(6488.TWO) 法人買賣」
        m = re.search(rf"\(({abbr}\.TWO?)\)", await page.title())
        if m is None or m.group(1) != symbol:
            continue
        # 沒有這項資料的股票(例如興櫃)會被導回報價頁
        if f"/{page_name}" not in page.url:
            continue

        # 同一頁可能有好幾個標題含 heading 的區塊 (例如大戶籌碼頁)，只取有表格的那個
        section = page.locator("section", has=page.locator("h2", has_text=heading)).filter(
            has=page.locator("li .table-row")
        )
        try:
            await section.wait_for(timeout=timeout_s * 1000)
            return symbol, section
        except Exception:
            continue
    raise RuntimeError(
        f"找不到代碼 {abbr} 的「{heading}」資料(興櫃股票 Yahoo 沒有提供)。已嘗試：\n" + "\n".join(tried)
    )


async def _click_until_selected(page, section, name, group, timeout_s):
    """點擊按鈕 name，直到它的背景變成淡藍色，且同一組 (group) 的其他按鈕都不是淡藍色
    (頁面剛載入時點擊可能沒反應，所以沒變色就重新點)"""
    btn = section.get_by_role("button", name=name, exact=True)
    deadline = time.monotonic() + timeout_s
    while True:
        await btn.click()
        # 按鈕 hover 時也會變淡藍色，先把滑鼠移開以免誤判
        await page.mouse.move(0, 0)
        await asyncio.sleep(1)
        bg = await section.evaluate(
            "sec => Object.fromEntries([...sec.querySelectorAll('button')]"
            ".map(b => [b.innerText.trim(), getComputedStyle(b).backgroundColor]))"
        )
        if bg.get(name) == SELECTED_BG and all(bg.get(o) != SELECTED_BG for o in group if o != name):
            return
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：「{name}」按鈕沒有變成選取狀態，按鈕底色：{bg}")


def _flatten_row(row):
    """有分組的欄位 (例如「融資」底下的 增減/餘額/使用率％) 讀進來會是 list，
    比照範本 CSV 在前面補一個空白的分組標題欄"""
    flat = []
    for cell in row:
        if isinstance(cell, list):
            flat += [""] + cell
        else:
            flat.append(cell)
    return flat


async def _wait_weekly_rows(section, n_cols, timeout_s):
    """讀取表格直到日期呈現「每週一筆」，回傳 (rows, dates)
    (表格可能延遲更新，所以輪詢到符合為止；只保留欄位數為 n_cols 的列)"""
    deadline = time.monotonic() + timeout_s
    while True:
        rows = await section.evaluate(
            "sec => [...sec.querySelectorAll('li .table-row')].map(r => [...r.children].map(c =>"
            " c.querySelector('li') ? [...c.querySelectorAll('li')].map(l => l.innerText.trim())"
            " : c.innerText.trim()))"
        )
        rows = [_flatten_row(r) for r in rows]
        rows = [r for r in rows if len(r) == n_cols]
        dates = [datetime.strptime(r[0], "%Y/%m/%d").date() for r in rows]
        if looks_weekly(dates):
            return rows, dates
        if time.monotonic() > deadline:
            raise RuntimeError("等待逾時：表格日期看起來不是「週」資料，請重新執行。")
        await asyncio.sleep(0.5)


def _to_week_df(rows, dates, columns, symbol):
    df = pd.DataFrame(rows, columns=columns)
    # 第一欄是日期，格式比照範本 (2026/10/8，不補 0)
    df[columns[0]] = [f"{d.year}/{d.month}/{d.day}" for d in dates]
    df.attrs["symbol"] = symbol  # 記下實際使用的代號 (例如 6488.TWO)，不會寫進 CSV
    return df


def run_in_own_loop(coro_fn, *args, **kwargs):
    """Jupyter 本身已有 event loop (Windows 上還是不支援子行程的 Selector loop)，
    所以另開一個執行緒、用自己的 event loop 跑 Playwright。"""
    result = {}

    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result["value"] = loop.run_until_complete(coro_fn(*args, **kwargs))
        except BaseException as e:
            result["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


def save_week_csv(df, abbr, file_tag, base_dir="."):
    """存成 ABBR/ABBR_{file_tag}_YYYYMMDD.csv (UTF-8 BOM、CRLF 換行，比照範本 CSV)"""
    folder = Path(base_dir) / abbr
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{abbr}_{file_tag}_{date.today():%Y%m%d}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig", lineterminator="\r\n")
    return path


# ===== 法人買賣超（週）爬蟲：函式定義 =====
# 網址 .../institutional-trading，點「週」後讀「法人逐日買賣超」表格

INSTITUTIONAL_COLUMNS = ["日期", "外資(張)", "投信(張)", "自營商(張)", "合計(張)", "外資籌碼", "漲跌幅(%)", "成交量"]


async def _scrape_institutional_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")
            symbol, section = await _open_section(page, abbr, "institutional-trading", "法人逐日買賣超", timeout_s)
            # 確認方法 1：「週」按鈕背景變成淡藍色，日/月/季按鈕不是淡藍色
            await _click_until_selected(page, section, "週", PERIOD_BUTTONS, timeout_s)
            # 確認方法 2：表格日期呈現「每週一筆」
            rows, dates = await _wait_weekly_rows(section, len(INSTITUTIONAL_COLUMNS), timeout_s)
        finally:
            await browser.close()
    return _to_week_df(rows, dates, INSTITUTIONAL_COLUMNS, symbol)


def fetch_institutional_trading_week(abbr, headless=True, timeout_s=30):
    """取得「法人逐日買賣超」→「週」的表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_institutional_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 資券餘額 / 借券賣出餘額（週）爬蟲：函式定義 =====
# 網址 .../margin，在「資券餘額逐日增減」依序切到兩個分頁、點「週」後讀表格

# 欄位比照 Margin_Week_11cols.csv、Securities_Lending_and_Borrowing_Week_8cols.csv
# 「融資」「融券」「借券賣出」是網頁上的分組標題，在 CSV 中是空白欄
MARGIN_COLUMNS = ["日期", "融資", "增減", "餘額", "使用率％", "融券", "增減", "餘額", "使用率％", "券資比", "資券互抵"]
LENDING_COLUMNS = ["日期", "融券", "增減", "餘額", "限額", "借券賣出", "增減", "餘額"]
MARGIN_TABS = {"資券餘額": MARGIN_COLUMNS, "借券賣出餘額": LENDING_COLUMNS}  # 分頁名稱: 欄位


async def _scrape_margin_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")
            symbol, section = await _open_section(page, abbr, "margin", "資券餘額逐日增減", timeout_s)
            dfs = []
            for tab, columns in MARGIN_TABS.items():
                # 分頁按鈕、「週」按鈕都要變成淡藍色
                await _click_until_selected(page, section, tab, MARGIN_TABS, timeout_s)
                await _click_until_selected(page, section, "週", PERIOD_BUTTONS, timeout_s)
                # 兩個分頁的欄位數不同 (11 / 8)，用欄位數篩選就不會讀到切換前的舊表格
                rows, dates = await _wait_weekly_rows(section, len(columns), timeout_s)
                dfs.append(_to_week_df(rows, dates, columns, symbol))
        finally:
            await browser.close()
    return tuple(dfs)


def fetch_margin_week(abbr, headless=True, timeout_s=30):
    """取得「資券餘額逐日增減」→「週」的兩個分頁，回傳 (資券餘額 DataFrame, 借券賣出餘額 DataFrame)"""
    return run_in_own_loop(_scrape_margin_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 大戶籌碼（週）爬蟲：函式定義 =====
# 網址 .../major-holders，表格本來就是每週一筆，不用點按鈕，只用表格日期確認

MAJOR_HOLDERS_COLUMNS = ["年度/日期", "外資籌碼", "大戶籌碼", "董監持股", "股價"]  # 比照 Major_Holders_Week_5cols.csv


async def _scrape_major_holders_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")
            # 區塊標題是「台積電(2330) 大戶籌碼」這種格式，用「大戶籌碼」比對
            symbol, section = await _open_section(page, abbr, "major-holders", "大戶籌碼", timeout_s)
            rows, dates = await _wait_weekly_rows(section, len(MAJOR_HOLDERS_COLUMNS), timeout_s)
        finally:
            await browser.close()
    return _to_week_df(rows, dates, MAJOR_HOLDERS_COLUMNS, symbol)


def fetch_major_holders_week(abbr, headless=True, timeout_s=30):
    """取得「大戶籌碼」的週表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_major_holders_week, abbr, headless=headless, timeout_s=timeout_s)

### 輸入公司代碼，開始蒐集資料

In [ ]:
# ===== 執行：輸入代碼 → 取得「週」法人買賣超、資券餘額、借券賣出餘額、大戶籌碼 → 存檔 =====
ABBR = ask_stock_code()

# 想看瀏覽器操作過程可加 headless=False
df_inst = fetch_institutional_trading_week(ABBR)
df_margin, df_lending = fetch_margin_week(ABBR)
df_major = fetch_major_holders_week(ABBR)

for df, file_tag in [
    (df_inst, "Institutional_Trading_Week_8cols"),
    (df_margin, "Margin_Week_11cols"),
    (df_lending, "Securities_Lending_and_Borrowing_Week_8cols"),
    (df_major, "Major_Holders_Week_5cols"),
]:
    csv_path = save_week_csv(df, ABBR, file_tag)
    print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path.resolve()}")
    display(df.head(3))